# 🚀 Lexi Research — Stage 1 (Stage A) Corrector Training on Colab

This notebook contains the complete, self-contained end-to-end pipeline to train **Stage 1 (Stage A - GEC Corrector)** using **Qwen3.5-0.8B** with QLoRA on Google Colab GPU.

### Optimal Configuration:
- **Batch Size**: `8` (Direct batch processing without accumulation latency)
- **Gradient Accumulation**: `1` (Immediate optimizer update per step)
- **Attention Implementation**: `sdpa` (PyTorch native FlashAttention-2 + Flash Linear Attention, 0s compile time)
- **Max Sequence Length**: `384` (Covers 96.86% of data, dropping only 3.14% extreme outliers)
- **Length-based Bucketing**: `group_by_length=true` (Grom similar sequence lengths per batch to eliminate padding tokens)
- **Fused CUDA Kernels**: `liger-kernel` (Triton fused cross-entropy + RMSNorm + SwiGLU) + `causal-conv1d` + `flash-linear-attention`
- **Fused Optimizer**: `adamw_torch_fused` (Native C++/CUDA fused AdamW)
- **Continuous Checkpointing**: Checkpoints stream directly to `/content/drive/MyDrive/lexi-runs/stage1_qwen08b`

## 1. Hardware & GPU Check

In [ ]:
!nvidia-smi

## 2. Mount Google Drive for Continuous Checkpointing

By setting the training output directory directly on Google Drive, all intermediate checkpoints (`checkpoint-*`) and final adapter weights are written straight to Drive in real-time.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/lexi-runs/stage1_qwen08b"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Checkpoints will stream directly to: {OUTPUT_DIR}")

## 3. Clone Repository & Install Dependencies

In [ ]:
# Clone repository & enter workspace
!git clone https://github.com/qninhdt/lexi-research.git /content/lexi-research
%cd /content/lexi-research

# ⚡ Single instant pip install command (pre-built binary wheels only, 0s C++ compilation)
!pip install -q --no-build-isolation -r requirements-colab.txt

### Verify Fused Kernels & CUDA Environment

In [ ]:
import os
import torch

# Optimize memory allocator and CUDA settings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

try:
    import flash_attn
    print("✅ flash-attn (FlashAttention-2): enabled")
except Exception as exc:
    print(f"⚠️ flash-attn: unavailable ({exc})")

try:
    import fla
    print("✅ flash-linear-attention: enabled")
except Exception as exc:
    print(f"⚠️ flash-linear-attention: unavailable ({exc})")

try:
    import causal_conv1d
    print("✅ causal-conv1d: enabled")
except Exception as exc:
    print(f"⚠️ causal-conv1d: unavailable ({exc})")

try:
    import liger_kernel
    print("✅ liger-kernel: enabled")
except Exception as exc:
    print(f"⚠️ liger-kernel: unavailable ({exc})")

## 4. Download & Import Stage 1 Dataset (W&I+LOCNESS v2.1)

Downloads Cambridge W&I+LOCNESS v2.1 corpus and converts M2 annotations into the `{original=>correction||tag}` parquet dataset.

In [ ]:
import os
import urllib.request
import tarfile

corpus_dir = "data/corpora/wi_locness"
if not os.path.exists(f"{corpus_dir}/m2"):
    url = "https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz"
    archive = "/content/wi_locness.tar.gz"
    print(f"Downloading {url}...")
    urllib.request.urlretrieve(url, archive)
    os.makedirs("data/corpora", exist_ok=True)
    with tarfile.open(archive) as tar:
        tar.extractall("data/corpora")
    if os.path.exists("data/corpora/wi+locness"):
        os.rename("data/corpora/wi+locness", corpus_dir)
    print("✅ Corpus extracted.")
else:
    print("Corpus already present.")

# Run GEC import to build train.parquet and val.parquet
!lexi data gec-import --corpus data/corpora/wi_locness --out data/gec

## 5. Pre-flight Smoke Test on GPU (`Qwen/Qwen3.5-0.8B`)

Quick sanity check over 50 rows to ensure quantization, kernels, and PEFT adapter attach properly.

In [ ]:
!lexi smoke --gpu --override train.base_model=Qwen/Qwen3.5-0.8B --override train.max_seq_len=384

## 6. Train Stage 1 (Corrector) SFT with Continuous Drive Checkpointing

- Target model: **`Qwen/Qwen3.5-0.8B`**
- Task: **`corrector`**
- Batch Size: **`8`** | Grad Accum: **`1`** (Maximum throughput, immediate parameter updates)
- Attention: **`sdpa`** (PyTorch native FlashAttention-2 without C++ compile overhead)
- Max Sequence Length: **`384`** (retains 96.86% of data, drops only 3.14% outliers)
- Length-based Bucketing: **`group_by_length=true`** (groups similar lengths per batch to eliminate padding waste)
- Liger Kernel: **`use_liger_kernel=true`** (Triton fused cross-entropy + RMSNorm + SwiGLU in GPU SRAM)
- Optimizer: **`adamw_torch_fused`** (Fused CUDA AdamW)
- Output: **`/content/drive/MyDrive/lexi-runs/stage1_qwen08b`** (saved continuously to Google Drive)
- Gradient Checkpointing: **Disabled** (for maximum speed on 0.8B parameter model)

In [ ]:
!lexi train sft \
  --train data/gec/train.parquet \
  --val data/gec/val.parquet \
  --band-config band_config.json \
  --output /content/drive/MyDrive/lexi-runs/stage1_qwen08b \
  --resume auto \
  --override train.base_model=Qwen/Qwen3.5-0.8B \
  --override train.task=corrector \
  --override train.thinking=off \
  --override train.epochs=1 \
  --override train.per_device_batch_size=8 \
  --override train.grad_accum=1 \
  --override train.max_seq_len=384 \
  --override train.group_by_length=true \
  --override train.use_liger_kernel=true \
  --override train.learning_rate=1.0e-4 \
  --override train.attn_implementation=sdpa \
  --override train.text_only=true \
  --override train.tf32=true \
  --override train.gradient_checkpointing=false

## 7. Inference & Quality Verification

Loads the trained adapter directly from Google Drive and verifies inline error markup on test sentences.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from lexi_research.format.parser import parse_correction
from lexi_research.train.corrector_prompt import render_corrector_prompt

model_id = "Qwen/Qwen3.5-0.8B"
adapter_path = "/content/drive/MyDrive/lexi-runs/stage1_qwen08b"

print(f"Loading base model {model_id} and adapter from {adapter_path}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

test_sentences = [
    "He speak very well in the meeting yesterday.",
    "I am look forward to see you next week.",
    "She have three cat in her house.",
    "The weather today is very beautifull.",
    "Although it was raining, but we still went outside."
]

print("\n--- Inference Verification ---")
for sent in test_sentences:
    messages = render_corrector_prompt(sent)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"\n[Original]  : {sent}")
    print(f"[Generated] : {generated}")
    try:
        edits, clean = parse_correction(sent, generated)
        print(f"[Parsed]    : {len(edits)} edit(s) -> '{clean}'")
        for e in edits:
            print(f"              - Tag: {e.tag} | Original: '{e.original}' -> Replacement: '{e.replacement}'")
    except Exception as e:
        print(f"[Parse Error]: {e}")